# 02 — First Alpha Simulation

This notebook walks through the complete flow:
1. Define an alpha expression using `AlphaConfig`
2. Submit it to WorldQuant Brain
3. Interpret the simulation results

## The fitness formula

```
fitness = sqrt(abs(returns) / max(turnover, 0.125)) * sharpe
```

**Target:** `fitness >= 1.0` with `sharpe >= 1.25`

- Higher Sharpe = better risk-adjusted returns
- Lower turnover = fewer trades = less friction
- Higher (absolute) returns = more alpha captured

## Alpha expression syntax

Expressions use Brain's operator language. Key operators:
- `ts_rank(x, n)` — rank of today's value in the past n-day window (0 to 1)
- `ts_zscore(x, n)` — z-score over past n days
- `ts_corr(x, y, n)` — rolling n-day correlation between x and y
- `ts_mean(x, n)` — rolling n-day mean
- `ts_delta(x, n)` — x today minus x n days ago
- `rank(x)` — cross-sectional rank (0 to 1) across all stocks today
- `group_rank(x, group)` — cross-sectional rank within a sector/industry group

In [19]:
import sys, importlib
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Force reload so any edits to wq/ take effect without a kernel restart
import wq.client, wq.alpha, wq.utils
importlib.reload(wq.client)
importlib.reload(wq.alpha)
importlib.reload(wq.utils)

from wq.client import get_client, simulate, fetch_alpha
from wq.alpha import AlphaConfig, compute_fitness, is_submittable
from wq.utils import parse_result, load_results

client = get_client()

Authenticated as jayesh.s@alumni.iitgn.ac.in


## Define your alpha

Start with a simple price momentum alpha: rank stocks by their price 20-day trend.

In [20]:
config = AlphaConfig(
    expression="ts_rank(close, 20)",
    universe="TOP3000",
    region="USA",
    neutralization="Subindustry",
    delay=1,
    decay=0,
    tags=["momentum", "price-volume", "baseline"],
    notes="Simple 20-day price momentum — baseline alpha",
)

print(config)
print(f'\nFingerprint: {config.fingerprint()}')
print(f'\nAPI payload preview:')
import json
print(json.dumps(config.to_payload(), indent=2))

AlphaConfig('ts_rank(close, 20)', universe=TOP3000, neutralization=Subindustry, delay=1, decay=0)

Fingerprint: 4a2b726a46fe1ebe

API payload preview:
{
  "type": "REGULAR",
  "settings": {
    "instrumentType": "EQUITY",
    "region": "USA",
    "universe": "TOP3000",
    "delay": 1,
    "decay": 0,
    "neutralization": "SUBINDUSTRY",
    "truncation": 0.08,
    "pasteurization": "ON",
    "unitHandling": "VERIFY",
    "nanHandling": "ON",
    "language": "FASTEXPR",
    "visualization": false
  },
  "regular": "ts_rank(close, 20)"
}


## Submit and wait for results

This typically takes 2–5 minutes. The result is automatically saved to `data/results.csv`.

In [21]:
result = simulate(client, config)
print('\nRaw result keys:', list(result.keys()))

Submitting: ts_rank(close, 20)
  sim_id=GhxVldmD5gV9U5aR8zPFhw — polling every 10s...
  status=COMPLETE

Raw result keys: ['id', 'type', 'settings', 'regular', 'status', 'alpha', 'is', 'os']


In [22]:
# If result is missing metrics (all None), fetch the alpha record directly.
# This happens when result was captured before the alpha-fetch fix was loaded.
def _enrich(session, r):
    alpha = r.get("alpha")
    if isinstance(alpha, str) and alpha and "is" not in r:
        alpha_data = fetch_alpha(session, alpha)
        for key in ("is", "os"):
            if key in alpha_data:
                r[key] = alpha_data[key]
        r["alpha"] = alpha_data
    return r

result    = _enrich(client, result)
result_v2 = _enrich(client, result_v2) if "result_v2" in dir() else result

print("is keys:", list((result.get("is") or {}).keys()))
print("sharpe :", result.get("is", {}).get("sharpe"))

is keys: ['pnl', 'bookSize', 'longCount', 'shortCount', 'turnover', 'returns', 'drawdown', 'margin', 'sharpe', 'fitness', 'startDate', 'checks']
sharpe : -1.42


## Interpret the results

In [23]:
metrics = parse_result(result)

print('=== In-sample metrics ===')
for k, v in metrics.items():
    if v is not None:
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Compute fitness manually to verify
sharpe   = metrics['is_sharpe']
returns  = metrics['is_returns']
turnover = metrics['is_turnover']

if all(v is not None for v in [sharpe, returns, turnover]):
    fitness = compute_fitness(sharpe, returns, turnover)
    print(f'\nComputed fitness: {fitness:.4f}')
    print(f'Sharpe threshold (>=1.25): {"PASS" if sharpe >= 1.25 else "FAIL"}')
    print(f'Submittable: {is_submittable(result)}')

=== In-sample metrics ===
  status: COMPLETE
  is_sharpe: -1.4200
  is_fitness: -0.7200
  is_returns: -0.1339
  is_turnover: 0.5218
  is_drawdown: 0.6926

Computed fitness: -0.7193
Sharpe threshold (>=1.25): FAIL
Submittable: False


## Try variations

Quickly test a few tweaks to the same alpha idea.

In [24]:
# Variation: neutralize at Industry level instead of Subindustry
config_v2 = AlphaConfig(
    expression="ts_rank(close, 20)",
    neutralization="Industry",
    tags=["momentum", "price-volume"],
    notes="ts_rank(close,20) with Industry neutralization",
)

result_v2 = simulate(client, config_v2)
metrics_v2 = parse_result(result_v2)

print('V2 (Industry neutralization):')
print(f"  Sharpe: {metrics_v2['is_sharpe']}  Fitness: {metrics_v2['is_fitness']}  Turnover: {metrics_v2['is_turnover']}")

Submitting: ts_rank(close, 20)
  sim_id=4lGrPYdNM57k9yG1eChNbHN8 — polling every 10s...
  status=COMPLETE
V2 (Industry neutralization):
  Sharpe: -1.3  Fitness: -0.69  Turnover: 0.4978


In [25]:
# Compare both results side by side
import pandas as pd

comparison = pd.DataFrame([
    {"alpha": "ts_rank(close,20) / Subindustry", **{k: v for k, v in parse_result(result).items() if k.startswith('is_')}},
    {"alpha": "ts_rank(close,20) / Industry",    **{k: v for k, v in parse_result(result_v2).items() if k.startswith('is_')}},
])
comparison

,alpha,is_sharpe,is_fitness,is_returns,is_turnover,is_drawdown
0,"ts_rank(close,20) / Subindustry",-1.42,-0.72,-0.1339,0.5218,0.6926
1,"ts_rank(close,20) / Industry",-1.30,-0.69,-0.1400,0.4978,0.7321


In [26]:
# Check what we've saved to the results log
from wq.utils import load_results
df_results = load_results()
print(f'Results log: {len(df_results)} rows')
df_results.tail(5)

Results log: 9 rows


,sim_id,status,expression,fingerprint,tags,universe,neutralization,delay,decay,is_sharpe,is_fitness,is_returns,is_turnover,is_drawdown,os_sharpe,os_fitness,os_returns,os_turnover,notes
4,hQRni8mj4E1cvHor7JwjCS,COMPLETE,"ts_rank(close, 20)",4a2b726a46fe1ebe,"[""momentum"", ""price-volume"", ""baseline""]",TOP3000,Subindustry,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simple 20-day price momentum — baseline alpha
5,apl8rbIr4wwa0hMWfZUS4M,COMPLETE,"ts_rank(close, 20)",4a2b726a46fe1ebe,"[""momentum"", ""price-volume"", ""baseline""]",TOP3000,Subindustry,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simple 20-day price momentum — baseline alpha
6,3nChXX8no4UNaBXsaAqRBV2,COMPLETE,"ts_rank(close, 20)",86dca3cced4f6915,"[""momentum"", ""price-volume""]",TOP3000,Industry,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"ts_rank(close,20) with Industry neutralization"
7,GhxVldmD5gV9U5aR8zPFhw,COMPLETE,"ts_rank(close, 20)",4a2b726a46fe1ebe,"[""momentum"", ""price-volume"", ""baseline""]",TOP3000,Subindustry,1,0,-1.42,-0.72,-0.1339,0.5218,0.6926,NaN,NaN,NaN,NaN,Simple 20-day price momentum — baseline alpha
8,4lGrPYdNM57k9yG1eChNbHN8,COMPLETE,"ts_rank(close, 20)",86dca3cced4f6915,"[""momentum"", ""price-volume""]",TOP3000,Industry,1,0,-1.30,-0.69,-0.1400,0.4978,0.7321,NaN,NaN,NaN,NaN,"ts_rank(close,20) with Industry neutralization"
